# Real-World Validation — Student Performance Assistant

This notebook is **isolated validation only**. It never retrains, overwrites, or modifies the production model or Streamlit app.

**Dataset:** UCI Student Performance, Portuguese course (`student-por.csv`), UCI dataset ID 320. UCI reports 649 records and notes that `G3` is the final-grade outcome and is strongly correlated with `G1` and `G2`.

Validation flow:

`UCI data → cleaning → EDA → target/risk definition → schema compatibility check → independent Random Forest benchmark → metrics → comparison with recorded production metrics`

We will **not fabricate missing production features** just to make the current model accept the UCI file.


In [ ]:
# ============================================================
# 1. SAFE PROJECT SETUP
# ============================================================

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Notebook is expected inside: project/notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

# Recovery if Antigravity launches the notebook from elsewhere.
if not (PROJECT_ROOT / "models").exists():
    for candidate in [
        Path.cwd().resolve(),
        Path.cwd().resolve().parent,
        Path.cwd().resolve().parent.parent
    ]:
        if (candidate / "models").exists() and (candidate / "data").exists():
            PROJECT_ROOT = candidate
            break

REPORT_DIR = PROJECT_ROOT / "reports" / "real_world_validation"
DATA_DIR = PROJECT_ROOT / "data" / "external"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Validation output:", REPORT_DIR)
print("Production model folder:", PROJECT_ROOT / "models")


In [ ]:
# ============================================================
# 2. LOAD UCI STUDENT PERFORMANCE DATA
# ============================================================

# If student-por.csv has already been downloaded into
# data/external/, it will be used.
#
# Otherwise this cell fetches UCI dataset 320.

local_file = DATA_DIR / "student-por.csv"

if local_file.exists():
    df_raw = pd.read_csv(local_file, sep=";")
    print("Loaded local file:", local_file)
else:
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError:
        raise ImportError(
            "Install the UCI helper first: pip install ucimlrepo"
        )

    dataset = fetch_ucirepo(id=320)

    X = dataset.data.features.copy()
    y = dataset.data.targets.copy()

    df_raw = pd.concat(
        [X.reset_index(drop=True), y.reset_index(drop=True)],
        axis=1
    )

    print("Loaded dataset directly from UCI.")

print("Shape:", df_raw.shape)
display(df_raw.head())


In [ ]:
# ============================================================
# 3. DATA UNDERSTANDING
# ============================================================

print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))

print("\nColumns:")
for col in df_raw.columns:
    print("-", col)

print("\nMissing values:")
display(
    df_raw.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nDuplicate rows:", df_raw.duplicated().sum())


In [ ]:
# ============================================================
# 4. CLEANING
# ============================================================

df = df_raw.copy()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
)

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", before - len(df))

# Convert numeric-looking fields.
for col in df.columns:
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().mean() >= 0.95:
        df[col] = converted

print("\nRemaining missing values:")
display(
    df.isna().sum()
    .loc[lambda x: x > 0]
    .to_frame("missing_count")
)

print("Clean shape:", df.shape)


In [ ]:
# ============================================================
# 5. DEFINE VALIDATION RISK TARGET
# ============================================================

# G3 is UCI's final grade on a 0–20 scale.
# It is the OUTCOME, not a predictor.
#
# Project-specific validation labels:
#   High   : G3 < 10
#   Medium : 10 <= G3 < 14
#   Low    : G3 >= 14
#
# These labels are created for this experiment and are NOT
# original UCI risk labels.

if "g3" not in df.columns:
    raise ValueError("G3 was not found in the UCI dataset.")

def grade_to_risk(g3):
    if g3 < 10:
        return "High"
    if g3 < 14:
        return "Medium"
    return "Low"

df["validation_risk"] = df["g3"].apply(grade_to_risk)

display(
    df["validation_risk"]
      .value_counts()
      .reindex(["High", "Medium", "Low"])
      .fillna(0)
      .astype(int)
      .to_frame("students")
)


In [ ]:
# ============================================================
# 6. EDA
# ============================================================

risk_counts = (
    df["validation_risk"]
      .value_counts()
      .reindex(["High", "Medium", "Low"])
      .fillna(0)
)

plt.figure(figsize=(7, 4))
risk_counts.plot(kind="bar")
plt.title("UCI Real-World Risk Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Students")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(df["g3"], bins=11)
plt.title("UCI Final Grade (G3) Distribution")
plt.xlabel("G3")
plt.ylabel("Students")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 7. EDA — LEARNING / PERFORMANCE VARIABLES
# ============================================================

candidate_cols = [
    "age", "studytime", "failures", "absences",
    "g1", "g2", "famrel", "freetime", "goout", "health"
]

available = [c for c in candidate_cols if c in df.columns]

display(df[available + ["g3"]].describe().T)

print("Mean values by validation risk:")
display(
    df.groupby("validation_risk")[available]
      .mean(numeric_only=True)
      .reindex(["High", "Medium", "Low"])
      .round(2)
)


In [ ]:
# ============================================================
# 8. READ-ONLY PRODUCTION MODEL SCHEMA CHECK
# ============================================================

metadata_candidates = [
    PROJECT_ROOT / "models" / "model_metadata.json",
    PROJECT_ROOT / "models" / "student_risk_model_metadata.json",
    PROJECT_ROOT / "models" / "metadata.json"
]

metadata_path = next(
    (p for p in metadata_candidates if p.exists()),
    None
)

production_features = []

if metadata_path:
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    production_features = metadata.get("features", [])

    print("Production metadata:", metadata_path)
    print("\nProduction features:")
    for feature in production_features:
        print("-", feature)

    common = [f for f in production_features if f in df.columns]
    missing = [f for f in production_features if f not in df.columns]

    print("\nCommon features:", common)

    print("\nProduction features missing from UCI:")
    for feature in missing:
        print("-", feature)
else:
    print("Production metadata JSON was not found.")


In [ ]:
# ============================================================
# 9. SAFETY DECISION
# ============================================================

if production_features:
    missing = [f for f in production_features if f not in df.columns]

    if missing:
        print("SAFE DECISION: DO NOT pass UCI data directly into the")
        print("current production model.")
        print()
        print("Reason: required production features are missing.")
        print("We will NOT invent those values.")
        print("We will NOT retrain or overwrite the production model.")
    else:
        print("All production features are present.")
        print("A direct read-only evaluation may be possible.")
else:
    print("No production feature metadata available.")
    print("Proceeding with an independent real-world benchmark.")

print("\nProduction model remains untouched.")


In [ ]:
# ============================================================
# 10. INDEPENDENT UCI BENCHMARK
# ============================================================

# IMPORTANT:
# This is a NEW model created only inside this notebook.
# It is NOT saved into models/ and does NOT replace our model.

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# G3 is the target. Remove it from predictors.
X_real = df.drop(columns=["g3", "validation_risk"])
y_real = df["validation_risk"]

# Early-warning setup:
# G1/G2 are prior-period grades and can be used in some settings,
# but UCI explicitly notes their strong correlation with G3.
# We remove them here for a stricter early-warning experiment.
X_real = X_real.drop(
    columns=[c for c in ["g1", "g2"] if c in X_real.columns]
)

X_train, X_test, y_train, y_test = train_test_split(
    X_real,
    y_real,
    test_size=0.20,
    random_state=42,
    stratify=y_real
)

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

validation_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

validation_pipeline.fit(X_train, y_train)

print("Independent UCI benchmark trained.")
print("Production model was not modified.")


In [ ]:
# ============================================================
# 11. EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

y_pred = validation_pipeline.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_weighted": precision_score(
        y_test, y_pred, average="weighted", zero_division=0
    ),
    "recall_weighted": recall_score(
        y_test, y_pred, average="weighted", zero_division=0
    ),
    "f1_weighted": f1_score(
        y_test, y_pred, average="weighted", zero_division=0
    ),
    "precision_macro": precision_score(
        y_test, y_pred, average="macro", zero_division=0
    ),
    "recall_macro": recall_score(
        y_test, y_pred, average="macro", zero_division=0
    ),
    "f1_macro": f1_score(
        y_test, y_pred, average="macro", zero_division=0
    )
}

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=["High", "Medium", "Low"],
        zero_division=0
    )
)


In [ ]:
# ============================================================
# 12. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["High", "Medium", "Low"]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["High", "Medium", "Low"]
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax)
plt.title("UCI Benchmark Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 13. INDEPENDENT MODEL FEATURE IMPORTANCE
# ============================================================

rf_model = validation_pipeline.named_steps["model"]
prep = validation_pipeline.named_steps["preprocessor"]

feature_names = prep.get_feature_names_out()

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .head(15)
)

display(importance_df)

plt.figure(figsize=(9, 6))
plt.barh(
    importance_df["feature"][::-1],
    importance_df["importance"][::-1]
)
plt.title("Top Features — Independent UCI Benchmark")
plt.xlabel("Random Forest Importance")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 14. COMPARE WITH RECORDED PRODUCTION METRICS
# ============================================================

# IMPORTANT:
# These are recorded values from the existing production model.
# This notebook does NOT recalculate or modify them.

production_metrics = {
    "accuracy": 0.7038,
    "precision_weighted": None,
    "recall_weighted": None,
    "f1_weighted": None
}

comparison = []

for metric_name, real_value in metrics.items():
    comparison.append({
        "metric": metric_name,
        "production_model_recorded": production_metrics.get(metric_name),
        "uci_independent_benchmark": real_value
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

comparison_df.to_csv(
    REPORT_DIR / "production_vs_uci_benchmark.csv",
    index=False
)

print("Saved:", REPORT_DIR / "production_vs_uci_benchmark.csv")


In [ ]:
# ============================================================
# 15. SAVE VALIDATION ARTIFACTS — NEVER models/
# ============================================================

results = {
    "dataset": "UCI Student Performance - Portuguese course",
    "uci_dataset_id": 320,
    "target": "G3",
    "validation_risk_definition": {
        "High": "G3 < 10",
        "Medium": "10 <= G3 < 14",
        "Low": "G3 >= 14"
    },
    "production_model_modified": False,
    "production_model_retrained": False,
    "production_model_overwritten": False,
    "metrics": metrics
}

with open(
    REPORT_DIR / "validation_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, indent=4)

df.to_csv(
    REPORT_DIR / "uci_student_por_cleaned.csv",
    index=False
)

print("Validation artifacts saved to:")
print(REPORT_DIR)


# 16. Interpretation

### Important distinction

This notebook contains **two separate things**:

**Production model**
- Your existing Random Forest.
- Loaded/read only when checking metadata.
- Never retrained.
- Never overwritten.
- Still used by Streamlit.

**UCI benchmark**
- A new Random Forest trained only inside this notebook.
- Uses UCI's own feature schema.
- Exists only to measure how a Random Forest performs on independent real-world student data.
- Not saved into `models/`.

### Why we do not force UCI into the production model

The UCI dataset does not contain all of the production features such as LMS activity, video completion, assignment completion, etc. Fabricating those values would make the evaluation unreliable.

Therefore, a low or high UCI benchmark score should be described as an **external benchmark**, not as a direct accuracy score of the production model.

For a stronger future validation study, the production model should eventually be retrained using a real-world dataset with a feature schema matching the application's inputs.
